# **OpenCV Project - Real Life Problem: Automated Fruit Ripeness & Sorting System**
## **Automated Fruit Ripeness, Quality Inspection & Sorting System**
### **Student Assignment & Presentation Notebook**

---

### **Project Overview & Real-World Motivation:**
In modern food supply chains, agricultural packaging plants, and supermarkets, inspecting fruit ripeness and sorting fruits manually is slow, labor-intensive, and prone to human errors.

**Our Computer Vision Solution:**  
Using fundamental **OpenCV operations** (as learned in `OpenCV_Lab.ipynb`), we build an end-to-end automated quality inspection pipeline that:
1. Ingests and inspects fruit images across multiple color representations (**BGR, Grayscale, HSV, LAB**).
2. Overlays a **Calibration Grid System** to measure pixel coordinates $(X, Y)$ on conveyor belts.
3. Applies **Geometric Transformations** (Resizing, Cropping, and Rotation).
4. Uses **Thresholding & Morphological Operations** (Opening & Closing) to isolate fruit bodies and remove camera noise.
5. Delineates object boundaries with **Sobel, Laplacian, and Canny Edge Detection**.
6. Measures physical dimensions using **Contours, Area, and Centroids**.
7. Detects and highlights **Unwanted Defect Dots / Blemishes on Unripe Fruit** using **Red Circles and Rectangles** across multiple backgrounds.
8. Implements a **4-Tier Ripeness Level System** (Level 1: Unripe to Level 4: Overripe) with color-coded status badges.
9. Classifies and sorts multiple fruit species (**Apple, Banana, Mango**).
10. Persists and saves all deliverables to disk for teacher verification.

### **Installing OpenCV & Libraries**

**Explanation for Presentation:**
- `opencv-python`: The core computer vision library for image manipulation, matrix operations, edge detection, and morphological transformations.
- `numpy`: High-performance multidimensional array library. Every image in OpenCV is represented as a NumPy `ndarray` of pixel values (0-255).
- `matplotlib`: Library for clean plotting and visualizations.

In [6]:
# using pip (if not already installed)
# !pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import os

# Dedicated folder to save all output images
output_dir = './output_results'
os.makedirs(output_dir, exist_ok=True)

print("OpenCV Version:", cv2.__version__)
print("Output results directory ready at:", output_dir)

OpenCV Version: 5.0.0
Output results directory ready at: ./output_results


### **Reading and Displaying Images with OpenCV**

**Explanation for Presentation:**
- `cv2.imread(path)`: Loads an image from the disk. By default, OpenCV reads color images in **BGR (Blue, Green, Red)** format rather than RGB.
- `cv2.imshow(winname, mat)`: Opens an interactive desktop window displaying the image matrix.
- `cv2.waitKey(delay)`: Waits for a keystroke. Passing `0` waits indefinitely, while passing milliseconds (e.g., `1000`) waits for 1 second.
- `cv2.destroyAllWindows()`: Closes all opened OpenCV GUI windows and frees GPU/CPU display memory.
- We load both the ripeness test image (`rip_unrip banana.jpg`) and the multi-fruit test image (`apple_banana_mango.jpg`).

In [7]:
import cv2

# Image paths from resources folder
banana_path = r'./resources/rip_unrip banana.jpg'
fruits_path = r'./resources/apple_banana_mango.jpg'

# Read images in BGR format
img_banana = cv2.imread(banana_path)
img_fruits = cv2.imread(fruits_path)

if img_banana is None or img_fruits is None:
    print("Error: Could not load images. Check the resources path!")
else:
    print(f"Banana Image Loaded: Shape = {img_banana.shape} (Height={img_banana.shape[0]}, Width={img_banana.shape[1]}, Channels={img_banana.shape[2]})")
    print(f"Fruits Image Loaded: Shape = {img_fruits.shape} (Height={img_fruits.shape[0]}, Width={img_fruits.shape[1]}, Channels={img_fruits.shape[2]})")

    # Display image in window
    cv2.imshow('Ripeness Test Image - Banana', img_banana)
    cv2.waitKey(1000)  # Wait for 1 second or key press
    cv2.destroyAllWindows()

Banana Image Loaded: Shape = (447, 447, 3) (Height=447, Width=447, Channels=3)
Fruits Image Loaded: Shape = (475, 646, 3) (Height=475, Width=646, Channels=3)


### **Writing and Saving Images in OpenCV**

**Explanation for Presentation:**
- `cv2.imwrite(path, img)`: Encodes and writes an image matrix to a file on disk.
- Saving interim and processed outputs is crucial in industrial quality inspection for auditing and batch tracking.

In [8]:
import cv2

# Save original input copies to the output results folder
cv2.imwrite('./output_results/01_input_banana.jpg', img_banana)
cv2.imwrite('./output_results/01_input_fruits.jpg', img_fruits)

print("Saved input images to ./output_results/ successfully!")

Saved input images to ./output_results/ successfully!


### **Draw a Line**

**Explanation for Presentation:**
- `cv2.line(img, pt1, pt2, color, thickness)`: Draws a line segment between two $(X, Y)$ coordinate tuples.
- **Color order:** OpenCV uses **(Blue, Green, Red)** tuples.
- We test drawing lines on two different backgrounds:
  1. A **Black Background canvas** (`np.zeros`) as practiced in lab exercises.
  2. Directly on the **Fruit Image** to create a conveyor lane dividing line.

In [13]:
import cv2
import numpy as np

# 1. Create blank black canvas (Height=300, Width=300, Channels=3)
canvas_line = np.zeros((300, 300, 3), dtype=np.uint8)

# Syntax: (image, start_point, end_point, color(BGR), thickness)
cv2.line(canvas_line, (50, 50), (250, 250), (0, 255, 255), 3)  # Yellow line

# 2. Draw conveyor lane dividing line directly on Fruit Image
fruit_line = img_banana.copy()
h, w, _ = fruit_line.shape
cv2.line(fruit_line, (w // 2, 0), (w // 2, h), (0, 0, 255), 3)  # Red vertical divider

cv2.imshow('Line on Black Background', canvas_line)
cv2.imshow('Conveyor Divider Line on Fruit', fruit_line)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Draw a Rectangle**

**Explanation for Presentation:**
- `cv2.rectangle(img, pt1, pt2, color, thickness)`: Draws a rectangle between top-left $(X_1, Y_1)$ and bottom-right $(X_2, Y_2)$.
- Setting `thickness = -1` creates a solid filled shape.
- Rectangles are the foundation of **Bounding Boxes** used to enclose detected fruits and inspection regions.

In [15]:
import cv2
import numpy as np

# 1. Green rectangle on black background
canvas_rect = np.zeros((300, 300, 3), dtype=np.uint8)
# Syntax: (image, top-left, bottom-right, color, thickness)
cv2.rectangle(canvas_rect, (50, 50), (250, 200), (0, 255, 0), 2)  # Green rectangle

# 2. Inspection Zone rectangle directly on Fruit Image
fruit_rect = img_banana.copy()
cv2.rectangle(fruit_rect, (30, 20), (w - 30, h - 20), (255, 120, 0), 3)  # Blue inspection boundary

cv2.imshow('Rectangle on Black Background', canvas_rect)
cv2.imshow('Inspection Zone on Fruit', fruit_rect)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Draw a Circle**

**Explanation for Presentation:**
- `cv2.circle(img, center, radius, color, thickness)`: Draws a circle.
- In fruit analysis, circles pinpoint fruit centroids, measure roundness, and flag blemishes or defect points.

In [ ]:
import cv2
import numpy as np

# 1. Red circle on black background
canvas_circle = np.zeros((300, 300, 3), dtype=np.uint8)
# Syntax: (image, center, radius, color, thickness)
cv2.circle(canvas_circle, (150, 150), 50, (0, 0, 255), -1)  # Red filled circle

# 2. Fruit centroid circle directly on Fruit Image
fruit_circle = img_banana.copy()
cv2.circle(fruit_circle, (w // 2, h // 2), 40, (0, 255, 255), 3)  # Yellow center circle

cv2.imshow('Circle on Black Background', canvas_circle)
cv2.imshow('Centroid Marker Circle on Fruit', fruit_circle)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Put Text on Image**

**Explanation for Presentation:**
- `cv2.putText(img, text, org, fontFace, fontScale, color, thickness)`: Renders text strings over image pixels.
- Vital in industrial systems for displaying fruit category, batch number, and inspection status on live camera monitors.

In [19]:
import cv2
import numpy as np

# 1. Text on black background
canvas_text = np.zeros((200, 500, 3), dtype=np.uint8)
# Syntax: (image, text, position, font, scale, color, thickness)
cv2.putText(canvas_text, "Fruit Inspection System", (30, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

# 2. Status text on Fruit Image
fruit_text = img_banana.copy()
cv2.putText(fruit_text, "Quality Status: PASSED", (25, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

cv2.imshow('Text on Black Background', canvas_text)
cv2.imshow('Status Text on Fruit', fruit_text)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Calibration Grid System (Coordinate Grid Overlay on Different Backgrounds)**

**Explanation for Presentation:**
- **Why an Inspection Grid is Useful:** Industrial sorting cameras require a coordinate calibration grid to verify where fruits are positioned on conveyor belts in pixel coordinates $(X, Y)$.
- We implement a custom grid system displaying horizontal and vertical grid lines with numeric coordinate ticks across:
  1. A **Black Background canvas** to isolate the pure coordinate system.
  2. The **Real Fruit Image** to map exact fruit boundaries on the conveyor belt.

In [21]:
import cv2
import numpy as np

def draw_inspection_grid(image, step=60, line_color=(80, 80, 80), text_color=(255, 255, 0)):
    """Draws a coordinate calibration grid over an image."""
    grid_canvas = image.copy()
    height, width, _ = grid_canvas.shape
    
    # Vertical grid lines (X axis)
    for x in range(0, width, step):
        cv2.line(grid_canvas, (x, 0), (x, height), line_color, 1)
        cv2.putText(grid_canvas, str(x), (x + 2, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)
        
    # Horizontal grid lines (Y axis)
    for y in range(0, height, step):
        cv2.line(grid_canvas, (0, y), (width, y), line_color, 1)
        cv2.putText(grid_canvas, str(y), (5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)
        
    return grid_canvas

# 1. Calibration Grid on Black Background
black_canvas = np.zeros_like(img_banana)
grid_black = draw_inspection_grid(black_canvas, step=70, line_color=(90, 90, 90), text_color=(0, 255, 255))

# 2. Calibration Grid on Real Fruit Image
grid_fruit = draw_inspection_grid(img_banana, step=70, line_color=(0, 255, 0), text_color=(255, 255, 255))

cv2.imshow('Calibration Grid on Black Background', grid_black)
cv2.imshow('Calibration Grid on Fruit Image', grid_fruit)
cv2.imwrite('./output_results/02_calibration_grid.jpg', grid_fruit)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Resizing**

**Explanation for Presentation:**
- `cv2.resize(src, dsize, fx, fy, interpolation)`: Standardizes image dimensions.
- **Fixed Resizing:** Setting exact pixel dimensions (e.g., $600 \times 400$).
- **Proportional Scaling:** Scaling by a ratio (e.g., $fx=0.5, fy=0.5$ for $50\%$ half-size).
- Downscaling images reduces processor lag and enables 30+ frames-per-second real-time camera inspection.

In [23]:

# Fixed size resizing (Width=600, Height=400)
resized_fixed = cv2.resize(img_banana, (600, 400))

# Proportional scaling (50% size)
scaled_half = cv2.resize(img_banana, None, fx=0.5, fy=0.5)

print("Original Shape :", img_banana.shape)
print("Resized Shape  :", resized_fixed.shape)
print("Scaled Shape   :", scaled_half.shape)

cv2.imshow('Original Fruit Image', img_banana)
cv2.imshow('Resized (600x400)', resized_fixed)
cv2.imshow('Scaled (50%)', scaled_half)
cv2.waitKey(0)
cv2.destroyAllWindows()

Original Shape : (447, 447, 3)
Resized Shape  : (400, 600, 3)
Scaled Shape   : (224, 224, 3)


### **Cropping**

**Explanation for Presentation:**
- Cropping is performed using 2D NumPy array slicing: `image[y1:y2, x1:x2]`.
- In our ripeness project, we crop the **Unripe Fruit Section (Green)** and the **Ripe Fruit Section (Yellow)** into separate Regions of Interest (ROI) for targeted quality inspection.

In [25]:
import cv2

# Standardized working image (600x400)
work_img = cv2.resize(img_banana, (600, 400))
h, w, _ = work_img.shape

# Crop Region: Left Half (Unripe Green Banana)
cropped_unripe = work_img[:, :w // 2]

# Crop Region: Right Half (Ripe Yellow Banana)
cropped_ripe = work_img[:, w // 2:]

cv2.imshow('Original Image', work_img)
cv2.imshow('Cropped Unripe Section', cropped_unripe)
cv2.imshow('Cropped Ripe Section', cropped_ripe)
cv2.imwrite('./output_results/03_cropped_unripe.jpg', cropped_unripe)
cv2.imwrite('./output_results/03_cropped_ripe.jpg', cropped_ripe)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Rotation**

**Explanation for Presentation:**
- `cv2.getRotationMatrix2D(center, angle, scale)`: Calculates the 2D affine rotation matrix around the image center.
- `cv2.warpAffine(src, M, dsize)`: Applies the matrix transformation to rotate the fruit.
- This corrects fruits that arrive tilted at arbitrary angles on conveyor rollers.

In [27]:
import cv2

center = (w // 2, h // 2)

# Rotate by 45 degrees
matrix_45 = cv2.getRotationMatrix2D(center, 45, 1.0)
rotated_45 = cv2.warpAffine(work_img, matrix_45, (w, h))

# Rotate by 90 degrees
matrix_90 = cv2.getRotationMatrix2D(center, 90, 1.0)
rotated_90 = cv2.warpAffine(work_img, matrix_90, (w, h))

cv2.imshow('Original Fruit', work_img)
cv2.imshow('Rotated 45 Degrees', rotated_45)
cv2.imshow('Rotated 90 Degrees', rotated_90)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **BGR to Grayscale**

**Explanation for Presentation:**
- `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)`: Converts a 3-channel color image to a single luminance/intensity channel.
- Reduces matrix memory by $66\%$, speeding up edge detection and thresholding.

In [28]:
import cv2

# Convert to grayscale
gray_fruit = cv2.cvtColor(work_img, cv2.COLOR_BGR2GRAY)

cv2.imshow('Original (BGR)', work_img)
cv2.imshow('Grayscale', gray_fruit)
cv2.imwrite('./output_results/04_grayscale.jpg', gray_fruit)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **BGR to RGB**

**Explanation for Presentation:**
- `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`: Swaps the Blue and Red channels.
- Standard display interfaces, web frameworks, and Matplotlib expect RGB format. If BGR is rendered directly in RGB viewers, reds become blues and blues become reds!

In [30]:
import cv2

# Convert BGR to RGB
rgb_fruit = cv2.cvtColor(work_img, cv2.COLOR_BGR2RGB)

cv2.imshow('Standard OpenCV (BGR)', work_img)
cv2.imshow('Converted RGB (Channels Inverted)', rgb_fruit)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **BGR to HSV (Hue-Saturation-Value)**

**Explanation for Presentation:**
- `cv2.cvtColor(img, cv2.COLOR_BGR2HSV)`: Converts to **Hue** (Pure color angle, 0-179 in OpenCV), **Saturation** (Color vibrancy, 0-255), and **Value** (Illumination/Brightness, 0-255).
- **Why HSV is the most critical operation in our project:** In RGB, shadow changes R, G, and B simultaneously. In HSV, color information is isolated strictly in **Hue**. This makes ripeness detection robust against changing warehouse lighting!

In [31]:
import cv2

# Convert to HSV
hsv_fruit = cv2.cvtColor(work_img, cv2.COLOR_BGR2HSV)

# Split individual channels
h_channel, s_channel, v_channel = cv2.split(hsv_fruit)

cv2.imshow('Original BGR', work_img)
cv2.imshow('HSV Representation', hsv_fruit)
cv2.imshow('Hue Channel (Color Tint)', h_channel)
cv2.imshow('Saturation Channel (Purity)', s_channel)
cv2.imshow('Value Channel (Brightness)', v_channel)
cv2.imwrite('./output_results/05_hue_channel.jpg', h_channel)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **BGR to LAB (CIELAB)**

**Explanation for Presentation:**
- `cv2.cvtColor(img, cv2.COLOR_BGR2LAB)`: Converts image into **L** (Lightness), **A** (Green to Red/Magenta axis), and **B** (Blue to Yellow axis).
- Designed to match human visual perception; useful for separating green skins from yellow skins.

In [32]:
import cv2

# Convert to LAB
lab_fruit = cv2.cvtColor(work_img, cv2.COLOR_BGR2LAB)
l_chan, a_chan, b_chan = cv2.split(lab_fruit)

cv2.imshow('Original BGR', work_img)
cv2.imshow('LAB Representation', lab_fruit)
cv2.imshow('L Channel (Lightness)', l_chan)
cv2.imshow('A Channel (Green-Magenta)', a_chan)
cv2.imshow('B Channel (Blue-Yellow)', b_chan)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Simple Thresholding**

**Explanation for Presentation:**
- `cv2.threshold(src, thresh, maxval, type)`: Compares each pixel against a fixed threshold value (e.g., 127).
- We test the 4 fundamental thresholding types taught in the lab:
  1. `cv2.THRESH_BINARY`: If pixel > 127, set to 255; else 0.
  2. `cv2.THRESH_BINARY_INV`: Inverted binary.
  3. `cv2.THRESH_TRUNC`: Pixels > 127 are clamped to 127.
  4. `cv2.THRESH_TOZERO`: Pixels < 127 are zeroed out.

In [34]:
import cv2

# Apply simple thresholding types
_, thresh_bin    = cv2.threshold(gray_fruit, 127, 255, cv2.THRESH_BINARY)
_, thresh_bininv = cv2.threshold(gray_fruit, 127, 255, cv2.THRESH_BINARY_INV)
_, thresh_trunc  = cv2.threshold(gray_fruit, 127, 255, cv2.THRESH_TRUNC)
_, thresh_tozero = cv2.threshold(gray_fruit, 127, 255, cv2.THRESH_TOZERO)

cv2.imshow('Grayscale', gray_fruit)
cv2.imshow('BINARY', thresh_bin)
cv2.imshow('BINARY_INV', thresh_bininv)
cv2.imshow('TRUNC', thresh_trunc)
cv2.imshow('TOZERO', thresh_tozero)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Adaptive Thresholding**

**Explanation for Presentation:**
- `cv2.adaptiveThreshold(src, maxValue, adaptiveMethod, thresholdType, blockSize, C)`: Calculates localized thresholds for small surrounding pixel neighborhoods.
- Handles non-uniform lighting and shadows across the fruit surface where global thresholding fails.

In [36]:
import cv2

# 1. Adaptive Mean Thresholding
thresh_adapt_mean = cv2.adaptiveThreshold(
    gray_fruit, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 15, 4
)

# 2. Adaptive Gaussian Thresholding
thresh_adapt_gauss = cv2.adaptiveThreshold(
    gray_fruit, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 4
)

cv2.imshow('Grayscale', gray_fruit)
cv2.imshow('Adaptive Mean', thresh_adapt_mean)
cv2.imshow('Adaptive Gaussian', thresh_adapt_gauss)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Otsu's Binarization**

**Explanation for Presentation:**
- `cv2.threshold(src, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)`: Automatically computes the optimal global threshold value based on the image histogram by minimizing intra-class variance.
- Eliminates manual threshold guess-work.

In [37]:
import cv2

# Apply Gaussian blur to clean noise before Otsu calculation
blurred_fruit = cv2.GaussianBlur(gray_fruit, (5, 5), 0)

# Otsu automatic thresholding
otsu_val, thresh_otsu = cv2.threshold(
    blurred_fruit, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

print(f"Otsu's Automatically Calculated Optimal Threshold: {otsu_val:.2f}")

cv2.imshow('Blurred Grayscale', blurred_fruit)
cv2.imshow(f"Otsu's Threshold ({otsu_val:.1f})", thresh_otsu)
cv2.imwrite('./output_results/06_otsu_threshold.jpg', thresh_otsu)
cv2.waitKey(1000)
cv2.destroyAllWindows()

Otsu's Automatically Calculated Optimal Threshold: 163.00


### **Canny Edge Detection**

**Explanation for Presentation:**
- `cv2.Canny(image, threshold1, threshold2)`: Multi-stage optimal edge detector.
- Thinning non-maximum suppression and hysteresis double-thresholding trace the crisp outer boundary of the fruit.

In [39]:
import cv2

# Apply Canny Edge Detector
canny_edges = cv2.Canny(blurred_fruit, 50, 150)

cv2.imshow('Grayscale Input', gray_fruit)
cv2.imshow('Canny Edges (Fruit Silhouette)', canny_edges)
cv2.imwrite('./output_results/07_canny_edges.jpg', canny_edges)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Sobel Operator**

**Explanation for Presentation:**
- `cv2.Sobel(src, ddepth, dx, dy, ksize)`: Calculates directional first derivatives.
- Separates vertical gradients (`dx=1, dy=0`) and horizontal gradients (`dx=0, dy=1`).

In [41]:
import cv2
import numpy as np

# Sobel derivatives
sobelx = cv2.Sobel(gray_fruit, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(gray_fruit, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = cv2.magnitude(sobelx, sobely)
sobel_8u = np.uint8(np.clip(sobel_mag, 0, 255))

cv2.imshow('Sobel X (Vertical Edges)', np.uint8(np.absolute(sobelx)))
cv2.imshow('Sobel Y (Horizontal Edges)', np.uint8(np.absolute(sobely)))
cv2.imshow('Sobel Combined Magnitude', sobel_8u)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Laplacian Operator**

**Explanation for Presentation:**
- `cv2.Laplacian(src, ddepth, ksize)`: Computes second-order spatial derivatives to detect rapid color transitions in all directions simultaneously.

In [43]:
import cv2
import numpy as np

laplacian = cv2.Laplacian(gray_fruit, cv2.CV_64F, ksize=3)
laplacian_8u = np.uint8(np.absolute(laplacian))

cv2.imshow('Grayscale Input', gray_fruit)
cv2.imshow('Laplacian Edge Representation', laplacian_8u)
cv2.waitKey(1000)
cv2.destroyAllWindows()

### **Erosion**

**Explanation for Presentation:**
- `cv2.erode(src, kernel, iterations)`: Shrinks foreground boundaries using a structuring kernel.
- Strips away thin detached noise points.

In [44]:
import cv2
import numpy as np

# 5x5 structuring kernel
kernel = np.ones((5, 5), np.uint8)
eroded_mask = cv2.erode(thresh_otsu, kernel, iterations=1)

cv2.imshow('Original Binary Mask', thresh_otsu)
cv2.imshow('Eroded Mask (Thinned)', eroded_mask)
cv2.waitKey(1000)
cv2.destroyAllWindows()

### **Dilation**

**Explanation for Presentation:**
- `cv2.dilate(src, kernel, iterations)`: Expands foreground boundaries.
- Thickens the fruit mask and bridges small gaps.

In [45]:
import cv2
import numpy as np

dilated_mask = cv2.dilate(thresh_otsu, kernel, iterations=1)

cv2.imshow('Original Binary Mask', thresh_otsu)
cv2.imshow('Dilated Mask (Thickened)', dilated_mask)
cv2.waitKey(1000)
cv2.destroyAllWindows()

### **Opening**

**Explanation for Presentation:**
- `cv2.morphologyEx(src, cv2.MORPH_OPEN, kernel)`: **Erosion followed by Dilation**.
- Eliminates external background salt-noise dots without shrinking the fruit body.

In [65]:
import cv2
import numpy as np

opened_mask = cv2.morphologyEx(thresh_otsu, cv2.MORPH_OPEN, kernel, iterations=2)

cv2.imshow('Original Binary Mask', thresh_otsu)
cv2.imshow('Opened Mask (External Noise Removed)', opened_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Closing**

**Explanation for Presentation:**
- `cv2.morphologyEx(src, cv2.MORPH_CLOSE, kernel)`: **Dilation followed by Erosion**.
- Fills internal holes and light-reflection gaps inside the fruit boundary.

In [66]:
import cv2
import numpy as np

closed_mask = cv2.morphologyEx(opened_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

cv2.imshow('Opened Mask', opened_mask)
cv2.imshow('Closed Mask (Clean Solid Fruit)', closed_mask)
cv2.imwrite('./output_results/08_morphology_clean.jpg', closed_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Drawing Contours**

**Explanation for Presentation:**
- `cv2.findContours(image, mode, method)`: Extracts closed curve boundaries representing individual fruits.
- `cv2.drawContours(image, contours, contourIdx, color, thickness)`: Renders boundaries over images.
- We compute **Contour Area** (`cv2.contourArea`) and **Perimeter** (`cv2.arcLength`) to filter out noise.

In [67]:
import cv2

# Find outer contours on morphological mask
contours, hierarchy = cv2.findContours(closed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

contour_canvas = work_img.copy()

# Filter valid fruit contours by minimum area
min_area = 1500
valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]

# Draw all fruit contours in Green
cv2.drawContours(contour_canvas, valid_contours, -1, (0, 255, 0), 3)

print(f"Total raw contours found: {len(contours)}")
print(f"Valid fruit contours (> {min_area}px): {len(valid_contours)}")

for i, cnt in enumerate(valid_contours):
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    print(f"Fruit #{i+1}: Area = {area:.1f}px, Perimeter = {perimeter:.1f}px")

cv2.imshow('Clean Mask', closed_mask)
cv2.imshow('Contours on Fruit', contour_canvas)
cv2.imwrite('./output_results/09_fruit_contours.jpg', contour_canvas)
cv2.waitKey(1000)
cv2.destroyAllWindows()

Total raw contours found: 8
Valid fruit contours (> 1500px): 4
Fruit #1: Area = 1787.5px, Perimeter = 307.5px
Fruit #2: Area = 2683.0px, Perimeter = 381.2px
Fruit #3: Area = 13282.5px, Perimeter = 617.9px
Fruit #4: Area = 8280.0px, Perimeter = 534.1px


### **Detection of Unwanted Dots / Blemishes on Unripe Fruit (Circles & Rectangles on Different Backgrounds)**

**Explanation for Presentation:**
- **Real-World Problem (Quality Inspection & Defect Analysis):**  
  Unripe fruits intended for shipping must be checked for brown spots, fungal blemishes, and bruising dots before packing.
- **Algorithm:**
  1. We isolate the unripe fruit crop (`cropped_unripe`).
  2. Dark defect dots on the green skin are segmented using inverse thresholding.
  3. Unwanted defect blobs ($15 < \text{Area} < 800\text{ px}$) are detected.
  4. On each unwanted dot, we draw:
     - A **Red Circle** (`cv2.circle`) around the centroid.
     - An **Orange/Red Rectangle** (`cv2.rectangle`) around the defect bounding box.
  5. We display this on both the **Real Fruit Background** and a **Black Canvas Background** so the teacher and viewers can clearly see each defect isolated!

In [69]:
import cv2
import numpy as np

# Work with the unripe fruit crop
unripe_bgr = cropped_unripe.copy()
unripe_gray = cv2.cvtColor(unripe_bgr, cv2.COLOR_BGR2GRAY)

# Threshold dark blemishes and defect dots
_, dot_thresh = cv2.threshold(unripe_gray, 95, 255, cv2.THRESH_BINARY_INV)

# Morphological opening with small 3x3 kernel
dot_kernel = np.ones((3, 3), np.uint8)
dot_clean = cv2.morphologyEx(dot_thresh, cv2.MORPH_OPEN, dot_kernel)

# Find defect contours
dot_contours, _ = cv2.findContours(dot_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 1. Display Canvas on Real Fruit Background
dots_on_fruit = unripe_bgr.copy()

# 2. Display Canvas on Black Background (for isolated defect analysis)
dots_on_black = np.zeros_like(unripe_bgr)

defect_count = 0
for cnt in dot_contours:
    area = cv2.contourArea(cnt)
    # Filter: Small unwanted defect spots (15 to 800 sq pixels)
    if 15 < area < 800:
        defect_count += 1
        x, y, w, h = cv2.boundingRect(cnt)
        cx, cy = x + w // 2, y + h // 2
        radius = max(w, h) // 2 + 4
        
        # --- A. Draw on Fruit Background ---
        cv2.rectangle(dots_on_fruit, (x - 2, y - 2), (x + w + 2, y + h + 2), (0, 165, 255), 2)  # Orange Rectangle
        cv2.circle(dots_on_fruit, (cx, cy), radius, (0, 0, 255), 2)                               # Red Circle
        cv2.putText(dots_on_fruit, f"#{defect_count}", (x - 4, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # --- B. Draw on Black Background Canvas ---
        cv2.rectangle(dots_on_black, (x - 2, y - 2), (x + w + 2, y + h + 2), (0, 255, 255), 1)
        cv2.circle(dots_on_black, (cx, cy), radius, (0, 0, 255), 2)
        cv2.putText(dots_on_black, f"Defect #{defect_count}", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

# Header labels
cv2.putText(dots_on_fruit, f"Unwanted Dots Found: {defect_count}", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
cv2.putText(dots_on_black, f"Defect Inspection: {defect_count} spots", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 1)

print(f"Total Unwanted Defect Dots Highlighted: {defect_count}")

cv2.imshow('Defect Dots on Fruit (Circle + Rectangle)', dots_on_fruit)
cv2.imshow('Defect Dots on Black Background Canvas', dots_on_black)
cv2.imwrite('./output_results/10_unwanted_dots_fruit.jpg', dots_on_fruit)
cv2.imwrite('./output_results/10_unwanted_dots_black_bg.jpg', dots_on_black)
cv2.waitKey(0)
cv2.destroyAllWindows()

Total Unwanted Defect Dots Highlighted: 18


### **Ripeness Level Grading System (Level 1 to Level 4)**

**Explanation for Presentation:**
- **Industrial Multi-Tier Grading:**  
  Commercial fruit packers do not simply use binary True/False. They divide fruits into **4 Ripeness Levels**:
  - **Level 1 (Hard Unripe / Raw):** Ripeness $< 25\%$ (Green color - Long distance transport)
  - **Level 2 (Semi-Ripe):** Ripeness $25\% - 50\%$ (Transitioning - Warehouse ripening)
  - **Level 3 (Optimal Ripe):** Ripeness $50\% - 80\%$ (Yellow color - Ready for supermarket shelves)
  - **Level 4 (Overripe):** Ripeness $> 80\%$ (Spotting - Processing for purees/juices)
- We compute the ratio of yellow pixels to green pixels within each fruit contour and overlay custom color-coded bounding boxes, centroid circles, and status badges!

In [70]:
import cv2
import numpy as np

# Define HSV color ranges for Green (Unripe) and Yellow (Ripe)
lower_green  = np.array([35, 40, 40], dtype=np.uint8)
upper_green  = np.array([85, 255, 255], dtype=np.uint8)
lower_yellow = np.array([15, 60, 60], dtype=np.uint8)
upper_yellow = np.array([34, 255, 255], dtype=np.uint8)

mask_green  = cv2.inRange(hsv_fruit, lower_green, upper_green)
mask_yellow = cv2.inRange(hsv_fruit, lower_yellow, upper_yellow)

graded_canvas = work_img.copy()

for i, cnt in enumerate(valid_contours):
    x, y, w, h = cv2.boundingRect(cnt)
    
    # Count yellow vs green pixels within the fruit bounding box
    yellow_px = cv2.countNonZero(mask_yellow[y:y+h, x:x+w])
    green_px  = cv2.countNonZero(mask_green[y:y+h, x:x+w])
    total_px  = yellow_px + green_px
    
    ripeness_pct = (yellow_px / total_px * 100) if total_px > 0 else 0.0
    
    # 4-Tier Ripeness Level logic
    if ripeness_pct < 25:
        level_label = "Level 1: Unripe (Raw)"
        badge_color = (0, 200, 0)     # Green BGR
    elif ripeness_pct < 50:
        level_label = "Level 2: Semi-Ripe"
        badge_color = (0, 255, 255)   # Yellow BGR
    elif ripeness_pct < 80:
        level_label = "Level 3: Optimal Ripe"
        badge_color = (0, 165, 255)   # Orange BGR
    else:
        level_label = "Level 4: Overripe"
        badge_color = (0, 0, 255)     # Red BGR
        
    # 1. Draw Bounding Box (cv2.rectangle)
    cv2.rectangle(graded_canvas, (x, y), (x + w, y + h), badge_color, 3)
    
    # 2. Draw Centroid Circle (cv2.circle)
    M = cv2.moments(cnt)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        cv2.circle(graded_canvas, (cx, cy), 6, (255, 255, 255), -1)
        
    # 3. Draw Status Badge with Text (cv2.putText)
    tag = f"{level_label} [{ripeness_pct:.0f}%]"
    (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    cv2.rectangle(graded_canvas, (x, y - th - 10), (x + tw + 10, y), badge_color, -1)
    cv2.putText(graded_canvas, tag, (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)
    
    print(f"Fruit #{i+1} Result -> {level_label} (Ripeness = {ripeness_pct:.1f}%)")

cv2.imshow('Multi-Level Ripeness Grading System', graded_canvas)
cv2.imwrite('./output_results/11_ripeness_levels.jpg', graded_canvas)
cv2.waitKey(0)
cv2.destroyAllWindows()

Fruit #1 Result -> Level 4: Overripe (Ripeness = 100.0%)
Fruit #2 Result -> Level 4: Overripe (Ripeness = 100.0%)
Fruit #3 Result -> Level 4: Overripe (Ripeness = 100.0%)
Fruit #4 Result -> Level 4: Overripe (Ripeness = 100.0%)


### **Multi-Fruit Sorting (Apple, Banana, Mango)**

**Explanation for Presentation:**
- Packaging warehouses frequently receive mixed fruit trays.
- We test our sorting pipeline on `resources/apple_banana_mango.jpg`.
- Using distinct HSV color ranges, we segment Apples (Red) and Bananas (Yellow), clean the masks via morphological operations, and place species classification bounding boxes.

In [73]:
import cv2
import numpy as np

# Resize multi-fruit image
fruits_work = cv2.resize(img_fruits, (600, 400))
hsv_multi = cv2.cvtColor(fruits_work, cv2.COLOR_BGR2HSV)

# Red mask for Apple (Red wraps around Hue 0 and 180)
mask_red1 = cv2.inRange(hsv_multi, np.array([0, 100, 70]), np.array([10, 255, 255]))
mask_red2 = cv2.inRange(hsv_multi, np.array([170, 100, 70]), np.array([180, 255, 255]))
mask_apple = cv2.bitwise_or(mask_red1, mask_red2)

# Yellow mask for Banana
mask_banana_multi = cv2.inRange(hsv_multi, np.array([20, 80, 80]), np.array([35, 255, 255]))

# Clean masks with morphological opening
m_kernel = np.ones((5, 5), np.uint8)
clean_apple  = cv2.morphologyEx(mask_apple, cv2.MORPH_OPEN, m_kernel)
clean_banana = cv2.morphologyEx(mask_banana_multi, cv2.MORPH_OPEN, m_kernel)

sorted_fruits = fruits_work.copy()

# Detect Apples
cnts_apple, _ = cv2.findContours(clean_apple, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
for c in cnts_apple:
    if cv2.contourArea(c) > 1000:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(sorted_fruits, (x, y), (x + w, y + h), (0, 0, 255), 3)  # Red BGR Box
        cv2.putText(sorted_fruits, "Apple", (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

# Detect Bananas
cnts_banana, _ = cv2.findContours(clean_banana, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
for c in cnts_banana:
    if cv2.contourArea(c) > 1000:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(sorted_fruits, (x, y), (x + w, y + h), (0, 255, 255), 3)  # Yellow BGR Box
        cv2.putText(sorted_fruits, "Banana", (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

cv2.imshow('Multi-Fruit Sorting (Apples vs Bananas)', sorted_fruits)
cv2.imwrite('./output_results/12_multi_fruit_sorting.jpg', sorted_fruits)
cv2.waitKey(0)
cv2.destroyAllWindows()

### **Summary of Generated Deliverables & Presentation Notes**

**Student Presentation Cheat Sheet:**
1. **Problem Solved:** Automated Fruit Ripeness & Multi-Fruit Sorting for smart agricultural quality control.
2. **OpenCV Lab Operations Integrated:**
   - Basic Operations: Image Loading, Saving, Lines, Rectangles, Circles, Text, Resizing, Cropping, and Rotation.
   - Calibration Grid: Coordinate inspection grid on both black background and real fruit background.
   - Color Spaces: Grayscale, RGB, HSV, and LAB.
   - Thresholding: Simple, Adaptive Gaussian, and Otsu's Binarization.
   - Gradients & Edges: Sobel, Laplacian, and Canny.
   - Morphology: Erosion, Dilation, Opening, and Closing.
   - Contours: Area, Perimeter, and Centroid calculations.
   - Quality Inspection: Detection of unwanted blemish dots on unripe fruits with red circles and rectangles.
   - Industrial Level System: 4-Tier Ripeness Classification (Level 1 to Level 4).
   - Multi-Fruit Classification: Automatic detection of Apples and Bananas.

In [74]:
import os

# Audit all saved deliverables
deliverables = os.listdir(output_dir)
print(f"\n🎉 Task Complete! Total {len(deliverables)} deliverables saved in '{output_dir}':")
for item in sorted(deliverables):
    print(f"   ✓ {item}")


🎉 Task Complete! Total 15 deliverables saved in './output_results':
   ✓ 01_input_banana.jpg
   ✓ 01_input_fruits.jpg
   ✓ 02_calibration_grid.jpg
   ✓ 03_cropped_ripe.jpg
   ✓ 03_cropped_unripe.jpg
   ✓ 04_grayscale.jpg
   ✓ 05_hue_channel.jpg
   ✓ 06_otsu_threshold.jpg
   ✓ 07_canny_edges.jpg
   ✓ 08_morphology_clean.jpg
   ✓ 09_fruit_contours.jpg
   ✓ 10_unwanted_dots_black_bg.jpg
   ✓ 10_unwanted_dots_fruit.jpg
   ✓ 11_ripeness_levels.jpg
   ✓ 12_multi_fruit_sorting.jpg
